# Data Mining I Comprehensive Study Guide

## Course Big Picture

Data mining is the process of turning raw data into useful patterns, relationships, predictions, or insights. This course focuses on **how real-world information is represented** and how that representation affects what we can discover.

The three major data representations in this course are:

| Representation         | Main Idea                              | Best For                                                 | Common Techniques                                            |
| ---------------------- | -------------------------------------- | -------------------------------------------------------- | ------------------------------------------------------------ |
| **Itemsets**           | Data as unordered collections of items | Market baskets, symptoms, tags, documents as word sets   | Frequent itemsets, association rules, Jaccard similarity     |
| **Vectors / Matrices** | Data as numeric coordinates or tables  | Ratings, user profiles, document-term matrices, features | Dot product, cosine similarity, Pearson correlation, PCA/SVD |
| **Sequences**          | Data as ordered lists                  | Text, DNA, clickstreams, course paths, event logs        | N-grams, edit distance, shingling, sequence alignment        |

The central idea: **the representation determines what information is preserved and what information is lost.**

---

# Week 1: Basic Concepts and Data Representations

## 1. Knowledge Discovery from Data

### Definition

**Knowledge Discovery from Data**, often called KDD, is the larger process of extracting useful knowledge from data.

A typical KDD pipeline:

1. Collect data
2. Clean data
3. Represent data formally
4. Mine patterns or compute similarities
5. Evaluate results
6. Interpret and apply findings

### Simple Example

Suppose a grocery store collects receipts.

Raw data:

```python
Receipt 1: milk, bread, eggs
Receipt 2: milk, bread
Receipt 3: bread, butter
Receipt 4: milk, eggs
```

Possible discoveries:

* Milk and bread are often bought together.
* Eggs frequently appear with milk.
* Bread is the most common item.

---

## 2. Major Data Mining Tasks

| Task                  | Definition                                      | Simple Example                          |
| --------------------- | ----------------------------------------------- | --------------------------------------- |
| **Association**       | Find items or events that occur together        | People who buy diapers also buy wipes   |
| **Retrieval**         | Find similar or relevant objects                | Search engine returns similar documents |
| **Classification**    | Assign objects to predefined categories         | Spam vs. not spam                       |
| **Prediction**        | Estimate a numeric or future value              | Predict house price                     |
| **Clustering**        | Group similar objects without predefined labels | Group customers by shopping behavior    |
| **Ranking**           | Order objects by relevance or importance        | Rank search results                     |
| **Outlier Detection** | Find unusual observations                       | Detect fraudulent transactions          |

---

## 3. Data Representation

### Why Representation Matters

The same real-world object can be represented in different ways, and each representation preserves different information.

Example sentence:

```python
"data mining data science"
```

As an **itemset**:

```python
{"data", "mining", "science"}
```

This loses frequency information because `"data"` appears twice but is only stored once.

As a **vector**:

```python
# dimensions: [data, mining, science]
[2, 1, 1]
```

This preserves frequency but loses word order.

As a **sequence**:

```python
["data", "mining", "data", "science"]
```

This preserves order and repetition.

---

# Representation 1: Itemsets

## Definition

An **itemset** is an unordered collection of unique items.

Example:

```python
basket = {"milk", "bread", "eggs"}
```

Itemsets are useful when order does not matter.

## Pros and Cons

| Pros                          | Cons                                        |
| ----------------------------- | ------------------------------------------- |
| Simple and compact            | Loses order                                 |
| Good for transactions         | Loses repeated counts                       |
| Useful for association mining | Cannot directly represent sequence behavior |

## Example Dataset

```python
transactions = [
    {"milk", "bread", "eggs"},
    {"milk", "bread"},
    {"bread", "butter"},
    {"milk", "eggs"},
    {"bread", "eggs"}
]
```

---

# Representation 2: Vectors and Matrices

## Definition

A **vector** represents an object as values across dimensions.

Example:

```python
# dimensions: [milk, bread, eggs, butter]
basket_vector = [1, 1, 1, 0]
```

A **matrix** is a collection of vectors.

Example:

```python
import pandas as pd

df = pd.DataFrame({
    "milk":  [1, 1, 0, 1, 0],
    "bread": [1, 1, 1, 0, 1],
    "eggs":  [1, 0, 0, 1, 1],
    "butter":[0, 0, 1, 0, 0]
})

df
```

Each row is a transaction. Each column is a feature.

## Pros and Cons

| Pros                                      | Cons                          |
| ----------------------------------------- | ----------------------------- |
| Works well with math and machine learning | Can be sparse with many zeros |
| Preserves counts or numeric values        | May lose order                |
| Supports distance and similarity measures | Requires fixed dimensions     |

---

# Representation 3: Sequences

## Definition

A **sequence** is an ordered list of items.

Example:

```python
course_path = ["Intro", "Stats", "Python", "ML"]
```

Sequences are useful when order matters.

## Pros and Cons

| Pros                                   | Cons                           |
| -------------------------------------- | ------------------------------ |
| Preserves order                        | More complex to compare        |
| Can represent text, DNA, paths, logs   | Slower similarity calculations |
| Useful for alignment and edit distance | Harder to summarize compactly  |

---

# Week 2: Itemset Data, Frequent Itemsets, and Similarity

## 1. Support

### Definition

**Support** measures how often an itemset appears in a dataset.

Formula:

```text
support(X) = number of transactions containing X / total number of transactions
```

### Example

```python
transactions = [
    {"milk", "bread", "eggs"},
    {"milk", "bread"},
    {"bread", "butter"},
    {"milk", "eggs"},
    {"bread", "eggs"}
]

def support(itemset, transactions):
    count = sum(1 for t in transactions if itemset.issubset(t))
    return count / len(transactions)

support({"milk"}, transactions)
```

Output:

```python
0.6
```

Milk appears in 3 out of 5 transactions.

---

## 2. Frequent Itemsets

### Definition

A **frequent itemset** is an itemset whose support is above a chosen minimum threshold.

Example:

```python
min_support = 0.4
```

An itemset is frequent if it appears in at least 40% of transactions.

### Code Example

```python
from itertools import combinations

def get_all_itemsets(transactions, k):
    # Get all unique items
    items = sorted(set().union(*transactions))
    
    # Generate all possible itemsets of size k
    return [set(combo) for combo in combinations(items, k)]

def frequent_itemsets(transactions, k, min_support):
    results = []
    
    for itemset in get_all_itemsets(transactions, k):
        supp = support(itemset, transactions)
        
        if supp >= min_support:
            results.append((itemset, supp))
            
    return results

frequent_itemsets(transactions, 2, 0.4)
```

Possible output:

```python
[({'bread', 'milk'}, 0.4), ({'eggs', 'milk'}, 0.4), ({'bread', 'eggs'}, 0.4)]
```

---

## 3. Apriori Algorithm

### Main Idea

The **Apriori algorithm** finds frequent itemsets efficiently using the **Apriori property**.

### Apriori Property

If an itemset is frequent, then all of its subsets must also be frequent.

Equivalently:

If an itemset is infrequent, then all larger itemsets containing it must also be infrequent.

### Simple Example

If:

```python
{"milk", "butter"}
```

is not frequent, then:

```python
{"milk", "butter", "eggs"}
```

cannot be frequent either.

This allows us to prune the search space.

---

## 4. Association Rules

### Definition

An **association rule** has the form:

```text
X → Y
```

Meaning: when X appears, Y often appears too.

Example:

```text
{milk} → {bread}
```

This means customers who buy milk often buy bread.

---

## 5. Confidence

### Definition

**Confidence** measures how often Y appears when X appears.

Formula:

```text
confidence(X → Y) = support(X ∪ Y) / support(X)
```

### Code Example

```python
def confidence(X, Y, transactions):
    return support(X.union(Y), transactions) / support(X, transactions)

confidence({"milk"}, {"bread"}, transactions)
```

Interpretation:

If confidence is `0.67`, then 67% of transactions with milk also include bread.

---

## 6. Lift

### Definition

**Lift** compares how often X and Y occur together versus how often we would expect if they were independent.

Formula:

```text
lift(X → Y) = confidence(X → Y) / support(Y)
```

### Interpretation

| Lift Value | Meaning                                   |
| ---------- | ----------------------------------------- |
| `lift > 1` | X and Y occur together more than expected |
| `lift = 1` | X and Y are independent                   |
| `lift < 1` | X and Y occur together less than expected |

### Code Example

```python
def lift(X, Y, transactions):
    return confidence(X, Y, transactions) / support(Y, transactions)

lift({"milk"}, {"bread"}, transactions)
```

---

## 7. Jaccard Similarity

### Definition

**Jaccard similarity** compares two sets by dividing the size of their intersection by the size of their union.

Formula:

```text
Jaccard(A, B) = |A ∩ B| / |A ∪ B|
```

### Simple Example

```python
A = {"milk", "bread", "eggs"}
B = {"milk", "bread", "butter"}

intersection = A.intersection(B)
union = A.union(B)

len(intersection) / len(union)
```

Output:

```python
0.5
```

They share 2 items out of 4 total unique items.

### Function

```python
def jaccard_similarity(set_a, set_b):
    # Similarity = shared items divided by total unique items
    return len(set_a.intersection(set_b)) / len(set_a.union(set_b))

jaccard_similarity(A, B)
```

### Jaccard Distance

```text
Jaccard distance = 1 - Jaccard similarity
```

```python
def jaccard_distance(set_a, set_b):
    return 1 - jaccard_similarity(set_a, set_b)
```

---

# Week 3: Vector and Matrix Data

## 1. Vector Representation

A vector stores numeric values across dimensions.

Example:

```python
# User ratings for [Restaurant A, Restaurant B, Restaurant C]
user_1 = [5, 4, 0]
user_2 = [4, 5, 0]
user_3 = [0, 1, 5]
```

The `0` may mean the user did not rate that restaurant.

---

## 2. Dot Product

### Definition

The **dot product** measures how much two vectors point in the same direction and how large they are.

Formula:

```text
A · B = a1b1 + a2b2 + ... + anbn
```

### Code Example

```python
import numpy as np

a = np.array([5, 4, 0])
b = np.array([4, 5, 0])

np.dot(a, b)
```

Output:

```python
40
```

Because:

```text
5*4 + 4*5 + 0*0 = 40
```

### Interpretation

A larger dot product can mean greater similarity, but it is affected by vector magnitude.

---

## 3. Euclidean Distance

### Definition

**Euclidean distance** measures straight-line distance between two vectors.

Formula:

```text
distance(A, B) = sqrt((a1-b1)^2 + (a2-b2)^2 + ... + (an-bn)^2)
```

### Code Example

```python
def euclidean_distance(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.sqrt(np.sum((a - b) ** 2))

euclidean_distance([5, 4, 0], [4, 5, 0])
```

Output:

```python
1.414
```

### Interpretation

Smaller distance means more similar.

---

## 4. Manhattan Distance

### Definition

**Manhattan distance** adds the absolute differences across dimensions.

Formula:

```text
distance(A, B) = |a1-b1| + |a2-b2| + ... + |an-bn|
```

### Code Example

```python
def manhattan_distance(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.sum(np.abs(a - b))

manhattan_distance([5, 4, 0], [4, 5, 0])
```

Output:

```python
2
```

### Interpretation

Manhattan distance is useful when movement happens dimension-by-dimension, like city blocks.

---

## 5. Cosine Similarity

### Definition

**Cosine similarity** measures the angle between two vectors, ignoring magnitude.

Formula:

```text
cosine(A, B) = (A · B) / (||A|| ||B||)
```

### Code Example

```python
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    
    numerator = np.dot(a, b)
    denominator = np.linalg.norm(a) * np.linalg.norm(b)
    
    return numerator / denominator

cosine_similarity([5, 4, 0], [4, 5, 0])
```

### Interpretation

| Cosine Value | Meaning                |
| ------------ | ---------------------- |
| `1`          | Same direction         |
| `0`          | Orthogonal / unrelated |
| `-1`         | Opposite direction     |

Cosine is especially useful for sparse high-dimensional data like text vectors or ratings.

---

## 6. Pearson Correlation

### Definition

**Pearson correlation** measures the linear relationship between two vectors after centering each vector around its mean.

It is similar to cosine similarity but applied after subtracting the mean.

### Code Example

```python
def pearson_correlation(a, b):
    a = np.array(a)
    b = np.array(b)
    
    a_centered = a - np.mean(a)
    b_centered = b - np.mean(b)
    
    return cosine_similarity(a_centered, b_centered)

pearson_correlation([5, 4, 1], [4, 5, 1])
```

### When Pearson Is Useful

Pearson is useful when users have different rating habits.

Example:

* User A rates everything high: `[5, 4, 5]`
* User B rates more strictly: `[3, 2, 3]`

They may still have similar preferences even though their rating scales differ.

---

## 7. Vector Similarity Comparison

| Measure             | Captures                       | Best When                             |
| ------------------- | ------------------------------ | ------------------------------------- |
| Dot Product         | Shared magnitude and direction | Larger values matter                  |
| Euclidean Distance  | Straight-line distance         | Magnitudes matter                     |
| Manhattan Distance  | Total absolute difference      | Dimensions are independently additive |
| Cosine Similarity   | Direction / pattern            | Magnitudes should be ignored          |
| Pearson Correlation | Centered pattern similarity    | Rating bias should be removed         |

---

## 8. Finding Similar Restaurants

Suppose restaurants are represented by user ratings.

```python
ratings = pd.DataFrame({
    "Restaurant_A": [5, 4, 0, 1],
    "Restaurant_B": [4, 5, 0, 1],
    "Restaurant_C": [0, 1, 5, 4],
    "Restaurant_D": [1, 0, 4, 5]
}, index=["User_1", "User_2", "User_3", "User_4"])

ratings
```

Each column is a restaurant. Each row is a user.

To compare restaurants:

```python
from sklearn.metrics.pairwise import cosine_similarity

# Transpose so restaurants become rows
restaurant_vectors = ratings.T

similarity_matrix = cosine_similarity(restaurant_vectors)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=restaurant_vectors.index,
    columns=restaurant_vectors.index
)

similarity_df
```

Find restaurants most similar to `Restaurant_A`:

```python
similarity_df["Restaurant_A"].sort_values(ascending=False)
```

---

## 9. Singular Value Decomposition

### Definition

**Singular Value Decomposition**, or **SVD**, factorizes a matrix into three matrices:

```text
A = UΣVᵀ
```

SVD is used for:

* Dimensionality reduction
* Noise reduction
* Recommendation systems
* Latent pattern discovery
* Matrix approximation

### Simple Intuition

A large ratings matrix may have hidden patterns, such as:

* Users who like casual restaurants
* Users who like expensive restaurants
* Users who prefer dessert places
* Restaurants with similar audience profiles

SVD helps uncover these hidden dimensions.

### Code Example

```python
import numpy as np

A = np.array([
    [5, 4, 0],
    [4, 5, 0],
    [0, 1, 5],
    [1, 0, 4]
])

U, S, VT = np.linalg.svd(A, full_matrices=False)

U, S, VT
```

### Low-Rank Approximation

```python
k = 2

U_k = U[:, :k]
S_k = np.diag(S[:k])
VT_k = VT[:k, :]

A_approx = U_k @ S_k @ VT_k

A_approx
```

This approximates the original matrix using only the top `k` latent dimensions.

---

# Week 4: Sequence Data, N-Grams, Edit Distance, Shingling, and Alignment

## 1. Sequence Data

### Definition

A **sequence** is an ordered list of elements.

Examples:

```python
text = ["data", "mining", "is", "useful"]

dna = ["A", "C", "G", "T"]

clickstream = ["home", "search", "product", "cart", "checkout"]
```

Sequences preserve order, unlike itemsets or basic vectors.

---

## 2. N-Grams

### Definition

An **n-gram** is a contiguous sequence of `n` items.

Example sentence:

```text
data mining is useful
```

### Unigrams

```text
data, mining, is, useful
```

### Bigrams

```text
data mining, mining is, is useful
```

### Trigrams

```text
data mining is, mining is useful
```

### Code Example

```python
def ngrams(sequence, n):
    return [tuple(sequence[i:i+n]) for i in range(len(sequence) - n + 1)]

tokens = ["data", "mining", "is", "useful"]

ngrams(tokens, 2)
```

Output:

```python
[('data', 'mining'), ('mining', 'is'), ('is', 'useful')]
```

---

## 3. Hamming Distance

### Definition

**Hamming distance** counts the number of positions where two equal-length sequences differ.

Example:

```text
A = "karolin"
B = "kathrin"
```

Differences:

```text
karolin
kathrin
  ^ ^^
```

### Code Example

```python
def hamming_distance(a, b):
    if len(a) != len(b):
        raise ValueError("Sequences must have the same length.")
    
    return sum(x != y for x, y in zip(a, b))

hamming_distance("karolin", "kathrin")
```

Output:

```python
3
```

### Limitation

Hamming distance only works for sequences of the same length.

---

## 4. Levenshtein Edit Distance

### Definition

**Levenshtein edit distance** measures how many operations are needed to transform one sequence into another.

Allowed operations:

1. Insert
2. Delete
3. Substitute

### Example

Transform:

```text
kitten → sitting
```

Steps:

```text
kitten → sitten   substitute k with s
sitten → sittin   substitute e with i
sittin → sitting  insert g
```

Edit distance = `3`.

---

## 5. Edit Distance Code

```python
def levenshtein_distance(s1, s2):
    # Create a matrix with one extra row/column for empty prefixes
    rows = len(s1) + 1
    cols = len(s2) + 1
    
    dp = [[0 for _ in range(cols)] for _ in range(rows)]
    
    # Cost of deleting characters from s1
    for i in range(rows):
        dp[i][0] = i
    
    # Cost of inserting characters into s1
    for j in range(cols):
        dp[0][j] = j
    
    # Fill the dynamic programming table
    for i in range(1, rows):
        for j in range(1, cols):
            
            if s1[i - 1] == s2[j - 1]:
                cost = 0
            else:
                cost = 1
            
            dp[i][j] = min(
                dp[i - 1][j] + 1,      # deletion
                dp[i][j - 1] + 1,      # insertion
                dp[i - 1][j - 1] + cost # substitution
            )
    
    return dp[-1][-1]

levenshtein_distance("kitten", "sitting")
```

Output:

```python
3
```

---

## 6. Why Edit Distance Is Expensive

Edit distance uses **dynamic programming** and compares many combinations of prefixes.

For strings of lengths `m` and `n`, the runtime is approximately:

```text
O(mn)
```

This can be slow for very long sequences or large datasets.

---

## 7. Shingling

### Definition

**Shingling** represents a document or sequence as a set of overlapping subsequences.

It is often used to detect near duplicates.

Example text:

```text
"data mining is useful"
```

With word shingles of size 2:

```python
{("data", "mining"), ("mining", "is"), ("is", "useful")}
```

### Code Example

```python
def shingles(text, k):
    tokens = text.lower().split()
    return set(tuple(tokens[i:i+k]) for i in range(len(tokens) - k + 1))

doc1 = "data mining is useful"
doc2 = "data mining is very useful"

shingles(doc1, 2)
```

Compare documents using Jaccard similarity:

```python
s1 = shingles(doc1, 2)
s2 = shingles(doc2, 2)

jaccard_similarity(s1, s2)
```

---

## 8. Why Shingling Is Faster Than Edit Distance

Edit distance compares sequences through a dynamic programming table.

Shingling converts sequences into sets and then uses Jaccard similarity.

This is often faster because set operations like intersection and union are efficient.

| Method              | Captures                  | Runtime Concern           |
| ------------------- | ------------------------- | ------------------------- |
| Edit Distance       | Exact transformation cost | Slower for long sequences |
| Shingling + Jaccard | Shared local subsequences | Faster and scalable       |

---

## 9. Near Duplicate Detection

### Definition

Near duplicate detection finds documents, strings, or sequences that are very similar but not exactly identical.

Examples:

```text
Doc 1: Data mining is useful for discovering patterns.
Doc 2: Data mining is very useful for discovering patterns.
```

These are not identical, but they are near duplicates.

### Code Example

```python
documents = [
    "data mining is useful for discovering patterns",
    "data mining is very useful for discovering patterns",
    "the weather today is sunny and warm"
]

def near_duplicates(docs, k=3, threshold=0.5):
    results = []
    
    for i in range(len(docs)):
        for j in range(i + 1, len(docs)):
            s1 = shingles(docs[i], k)
            s2 = shingles(docs[j], k)
            sim = jaccard_similarity(s1, s2)
            
            if sim >= threshold:
                results.append((i, j, sim))
                
    return results

near_duplicates(documents, k=2, threshold=0.4)
```

---

## 10. Sequence Alignment

### Definition

**Sequence alignment** compares two sequences by lining them up and allowing gaps.

It is commonly used in:

* DNA analysis
* Protein comparison
* Text comparison
* Path comparison

Example:

```text
A C G T
A - G T
```

The gap represents an insertion or deletion.

### Relationship to Edit Distance

Both edit distance and sequence alignment compare ordered sequences. Edit distance focuses on the minimum cost of edits. Sequence alignment often focuses on the best matching arrangement using match, mismatch, and gap scores.

---

# Key Formulas

## Support

```text
support(X) = count(transactions containing X) / total transactions
```

## Confidence

```text
confidence(X → Y) = support(X ∪ Y) / support(X)
```

## Lift

```text
lift(X → Y) = confidence(X → Y) / support(Y)
```

## Jaccard Similarity

```text
Jaccard(A, B) = |A ∩ B| / |A ∪ B|
```

## Jaccard Distance

```text
Jaccard distance = 1 - Jaccard similarity
```

## Dot Product

```text
A · B = Σ ai bi
```

## Euclidean Distance

```text
sqrt(Σ(ai - bi)^2)
```

## Manhattan Distance

```text
Σ |ai - bi|
```

## Cosine Similarity

```text
(A · B) / (||A|| ||B||)
```

## Levenshtein Distance

```text
minimum number of insertions, deletions, and substitutions
```

---

# Common Exam and Assignment Concepts

## Itemsets vs. Vectors vs. Sequences

| Question                           | Answer                                 |
| ---------------------------------- | -------------------------------------- |
| What does an itemset preserve?     | Which unique items are present         |
| What does an itemset lose?         | Order and repeated counts              |
| What does a vector preserve?       | Numeric values across fixed dimensions |
| What does a vector lose?           | Usually order                          |
| What does a sequence preserve?     | Order and repetition                   |
| What is the downside of sequences? | More expensive to compare              |

---

## Similarity vs. Distance

| Concept    | Meaning                 |
| ---------- | ----------------------- |
| Similarity | Higher means more alike |
| Distance   | Lower means more alike  |

Examples:

```text
Cosine similarity: higher is more similar
Euclidean distance: lower is more similar
Jaccard similarity: higher is more similar
Jaccard distance: lower is more similar
```

---

## Pattern Extraction vs. Similarity

| Method                 | Goal                      | Example                        |
| ---------------------- | ------------------------- | ------------------------------ |
| Pattern extraction     | Find recurring structures | Frequent itemsets              |
| Similarity measurement | Compare objects           | Jaccard, cosine, edit distance |

### Example

Pattern extraction asks:

```text
Which items often appear together?
```

Similarity asks:

```text
How similar are these two baskets, users, restaurants, or sequences?
```

---

# Mini Python Reference

## Itemset Support

```python
def support(itemset, transactions):
    return sum(itemset.issubset(t) for t in transactions) / len(transactions)
```

## Jaccard Similarity

```python
def jaccard_similarity(set_a, set_b):
    return len(set_a.intersection(set_b)) / len(set_a.union(set_b))
```

## Dot Product

```python
np.dot(a, b)
```

## Euclidean Distance

```python
np.sqrt(np.sum((a - b) ** 2))
```

## Manhattan Distance

```python
np.sum(np.abs(a - b))
```

## Cosine Similarity

```python
np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))
```

## Pearson Correlation

```python
np.corrcoef(a, b)[0, 1]
```

## N-Grams

```python
def ngrams(sequence, n):
    return [tuple(sequence[i:i+n]) for i in range(len(sequence) - n + 1)]
```

## Shingles

```python
def shingles(text, k):
    tokens = text.lower().split()
    return set(tuple(tokens[i:i+k]) for i in range(len(tokens) - k + 1))
```

## Hamming Distance

```python
def hamming_distance(a, b):
    return sum(x != y for x, y in zip(a, b))
```

## Levenshtein Distance

```python
def levenshtein_distance(s1, s2):
    rows = len(s1) + 1
    cols = len(s2) + 1
    dp = [[0] * cols for _ in range(rows)]

    for i in range(rows):
        dp[i][0] = i

    for j in range(cols):
        dp[0][j] = j

    for i in range(1, rows):
        for j in range(1, cols):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1

            dp[i][j] = min(
                dp[i - 1][j] + 1,
                dp[i][j - 1] + 1,
                dp[i - 1][j - 1] + cost
            )

    return dp[-1][-1]
```

---

# Study Checklist

## Week 1

You should be able to:

* Define data mining and knowledge discovery.
* Explain common data mining tasks.
* Compare itemsets, vectors, matrices, and sequences.
* Explain what information is lost when changing representations.
* Give real-world examples of each representation.

## Week 2

You should be able to:

* Define support, confidence, and lift.
* Identify frequent itemsets.
* Explain the Apriori property.
* Compute Jaccard similarity and distance.
* Explain association rules.

## Week 3

You should be able to:

* Represent objects as vectors or matrices.
* Compute dot product, Euclidean distance, Manhattan distance, cosine similarity, and Pearson correlation.
* Explain when each similarity or distance measure is useful.
* Use matrix representations for recommendation-style problems.
* Explain the purpose of SVD.

## Week 4

You should be able to:

* Define sequences and n-grams.
* Compute Hamming distance.
* Explain Levenshtein edit distance.
* Understand dynamic programming for edit distance.
* Use shingling for near duplicate detection.
* Explain how sequence alignment relates to edit distance.

---

# Quick Concept Map

```text
Data Mining
│
├── Data Representations
│   ├── Itemsets
│   ├── Vectors / Matrices
│   └── Sequences
│
├── Pattern Mining
│   ├── Frequent Itemsets
│   ├── Association Rules
│   ├── Support
│   ├── Confidence
│   └── Lift
│
├── Similarity and Distance
│   ├── Jaccard Similarity
│   ├── Dot Product
│   ├── Euclidean Distance
│   ├── Manhattan Distance
│   ├── Cosine Similarity
│   ├── Pearson Correlation
│   └── Edit Distance
│
└── Applications
    ├── Market Basket Analysis
    ├── Recommendation Systems
    ├── Search and Retrieval
    ├── Near Duplicate Detection
    ├── Sequence Alignment
    └── Classification / Clustering / Ranking
```

---

# High-Yield Exam Reminders

1. **Itemsets lose order and duplicates.**
2. **Vectors require fixed dimensions.**
3. **Sequences preserve order but are more expensive to compare.**
4. **Support measures frequency.**
5. **Confidence measures conditional likelihood.**
6. **Lift adjusts for baseline frequency.**
7. **Jaccard similarity compares overlap between sets.**
8. **Cosine similarity focuses on direction, not magnitude.**
9. **Pearson correlation is cosine similarity after mean-centering.**
10. **Edit distance measures transformation cost.**
11. **Shingling turns sequences/documents into sets of subsequences.**
12. **SVD finds lower-dimensional latent structure in matrices.**
